In [ ]:
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/SantanderCS'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
import numpy as np
import pandas as pd

from sklearn.ensemble         import RandomForestClassifier, StackingClassifier
from sklearn.linear_model     import LogisticRegression
from sklearn.model_selection  import StratifiedKFold
from sklearn.preprocessing    import StandardScaler
from sklearn.metrics          import roc_auc_score, recall_score, f1_score
from xgboost                  import XGBClassifier
from lightgbm                 import LGBMClassifier

from utils.preprocessing        import load_data, split_features_target, scale_data, data_split, remove_zero_columns2
from utils.user_utils           import get_model_train_eval
from utils.feature_engineering  import drop_highly_correlated_features


In [ ]:
def analyze_skewness(df, threshold=3.0):
    """
    데이터프레임의 왜도를 분석하고 log 변환 필요 여부 판단
    
    Args:
        df : pd.DataFrame
        threshold : float, 왜도 기준값
    """
    skewness = df.skew().sort_values(ascending=False)
    
    print(f"{'='*80}")
    print(f"왜도(Skewness) 분석")
    print(f"{'='*80}")
    print(f"전체 컬럼 수: {len(df.columns)}")
    print(f"왜도 > {threshold}: {len(skewness[abs(skewness) > threshold])}개")
    print(f"왜도 > 5: {len(skewness[abs(skewness) > 5])}개")
    print(f"왜도 > 10: {len(skewness[abs(skewness) > 10])}개")
    
    print(f"\n상위 20개 컬럼:")
    print(f"{'-'*80}")
    print(f"{'순위':>4} {'컬럼명':20} {'왜도':>10} {'최소값':>12} {'최대값':>12} {'Log변환':>10}")
    print(f"{'-'*80}")
    
    for i, (col, skew_val) in enumerate(skewness.head(20).items(), 1):
        min_val = df[col].min()
        max_val = df[col].max()
        log_ok = "가능" if min_val >= 0 else "불가(음수)"
        
        print(f"{i:4d} {col:20s} {skew_val:10.2f} {min_val:12.2f} {max_val:12.2f} {log_ok:>10}")
    
    print(f"{'='*80}\n")
    
    # 음수 값 분석
    negative_cols = [col for col in df.columns if (df[col] < 0).any()]
    print(f"음수 값이 있는 컬럼: {len(negative_cols)}개")
    if len(negative_cols) > 0:
        print(f"  예시: {negative_cols[:5]}")




In [ ]:
# 사용
train, test = load_data()
# X_features, y_labels = split_features_target(train)
# analyze_skewness(X_features, threshold=3.0)

In [ ]:
# 먼저 분석해보고
analyze_skewness(X_features)

# 왜도가 큰 컬럼이 많다면 실험
X_train1, X_val1, y_train1, y_val1 = santander_base_job2(isScaled=True, isSplit=True, apply_log=False)
X_train2, X_val2, y_train2, y_val2 = santander_base_job2(isScaled=True, isSplit=True, apply_log=True)

# 성능 비교 후 결정!